# MARV — SmolLM2 tool-calling vindex probe (T4)

Compares a base SmolLM2-135M-Instruct against a tool-call fine-tune of the
same size, plus the larger 1.7B-Instruct (official tool-calling support),
using MARV's gate-KNN + logit-lens + per-feature diff, scoped to fit in a
T4's 16GB.

Runtime: **T4 GPU** (Runtime > Change runtime type > T4).

In [ ]:
!pip install -q transformers accelerate numpy
!git clone -q https://github.com/thebnbrkr/marv.git /content/marv
%cd /content/marv
!pip install -q -e .

## Load the two 135M checkpoints (base vs. tool-tuned)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

BASE_135M = "HuggingFaceTB/SmolLM2-135M-Instruct"
TUNED_135M = "gvij/SmolLM2-135M-Function-Calling"

base_tok = AutoTokenizer.from_pretrained(BASE_135M)
base_model = AutoModelForCausalLM.from_pretrained(BASE_135M, torch_dtype=torch.float16).to(device).eval()

tuned_tok = AutoTokenizer.from_pretrained(TUNED_135M)
tuned_model = AutoModelForCausalLM.from_pretrained(TUNED_135M, torch_dtype=torch.float16).to(device).eval()

## Extract vindex-lite for both

In [ ]:
from marv.extract import extract

vindex_base = extract(base_model, model_name=BASE_135M)
vindex_tuned = extract(tuned_model, model_name=TUNED_135M)
print(vindex_base.num_layers, "layers,", vindex_base.hidden_size, "hidden")

## Concept probe: label firing features in each checkpoint

Each word is embedded in a short template sentence and run through the live model (a contextual hidden state, not a bare embedding row), then differenced against the template alone -- small transformers have a dominant, roughly prompt-invariant direction ("massive activation"/outlier channel) that otherwise swamps the query regardless of content. What's left after differencing is KNN'd over a *late* layer's gate rows (logit lens only reliably decodes to legible tokens in roughly the last third of the model) and labeled by what each firing feature promotes.

In [ ]:
from marv.toolcall import describe_prompt

PROBE_WORDS = ["weather", "function", "France"]
PROBE_TEMPLATE = "I want to talk about {word}"
PROBE_BASELINE = "I want to talk about"

def concept_probe(vindex, model, tok, label):
    late_layers = sorted(set(round(vindex.num_layers * f) for f in (0.7, 0.85)))
    late_layers = [l for l in late_layers if l < vindex.num_layers]
    print(f"--- {label} ---")
    for word in PROBE_WORDS:
        prompt = PROBE_TEMPLATE.format(word=word)
        hits = describe_prompt(
            vindex, model, tok, prompt, layers=late_layers, k_features=3, k_tokens=3,
            device=device, baseline_prompt=PROBE_BASELINE,
        )
        print(f"'{word}':")
        for layer, features in hits.items():
            for feature_idx, sim, tok_ids, logits in features:
                words_out = tok.batch_decode([[t] for t in tok_ids])
                print(f"  L{layer} f{feature_idx} (sim={sim:.2f}) -> {words_out}")

concept_probe(vindex_base, base_model, base_tok, "base")
concept_probe(vindex_tuned, tuned_model, tuned_tok, "tool-tuned")

## Polysemanticity heatmap: factual vs. instruction vs. coding features

Not everything is about tool-calling -- this asks a broader question: does
the model have genuinely distinct feature populations for different kinds of
knowledge, or do the same handful of neurons fire for everything (polysemantic
features)? Each word is embedded in the same differenced-template probe as
above; the heatmap's columns are the union of every word's top firing
features at one layer, so a column bright across multiple categories'
rows is a polysemantic feature, and a column bright only within one
category is concept-specific.

In [ ]:
from marv.heatmap import activation_matrix, plot_heatmap, polysemantic_features

categories = {
    "factual": ["Paris", "France", "capital"],
    "instruction": ["summarize", "explain"],
    "coding": ["code", "function", "return"],
}
late_layer = round(vindex_base.num_layers * 0.85)

am = activation_matrix(vindex_base, base_model, base_tok, categories, layer=late_layer, top_k_per_word=10, device=device)
fig = plot_heatmap(am, title=f"{BASE_135M}: feature activation by category")
fig

poly = polysemantic_features(am, threshold=0.15, min_categories=2)
print("polysemantic features (fire across >=2 categories):")
for feature_id, cats in sorted(poly.items()):
    print(f"  f{feature_id} -> {cats}")

## DIFF: which features moved most during tool-call fine-tuning

This is MARV's version-control primitive -- a ranked, sparse list of exactly
which FFN neurons changed, instead of a dense weight delta.

In [ ]:
from marv.diff import diff, most_changed

deltas = diff(vindex_base, vindex_tuned)
top = most_changed(deltas, k=15)
for d in top:
    print(f"L{d.layer} f{d.feature_idx}: gate_cos={d.gate_cos_sim:.3f} down_cos={d.down_cos_sim:.3f} norm_ratio={d.gate_norm_ratio:.2f}")

In [ ]:
# Label what the most-changed features promote, before vs. after
from marv.probe import describe_feature

for d in top[:5]:
    before = describe_feature(vindex_base, d.layer, d.feature_idx, k=3)
    after = describe_feature(vindex_tuned, d.layer, d.feature_idx, k=3)
    print(f"L{d.layer} f{d.feature_idx}")
    print("  before:", base_tok.batch_decode([[t] for t in before[0]]))
    print("  after: ", tuned_tok.batch_decode([[t] for t in after[0]]))

## Tool-call decision point: does the logit-lens answer flip?

Feed a prompt that should trigger a tool call, capture the residual stream at several *late* layers in both checkpoints (logit lens is noisy in early/mid layers), and see what each one "wants to say" at each depth.

In [ ]:
from marv.toolcall import DEFAULT_TOOL_PROMPTS, compare_tool_prompt, hidden_states_at_layers

start = max(1, round(vindex_base.num_layers * 0.66))
probe_layers = list(range(start, vindex_base.num_layers, 3))

for prompt in DEFAULT_TOOL_PROMPTS:
    print("\nprompt:", prompt)
    h_base = hidden_states_at_layers(base_model, base_tok, prompt, probe_layers, device=device)
    h_tuned = hidden_states_at_layers(tuned_model, tuned_tok, prompt, probe_layers, device=device)
    comparison = compare_tool_prompt(vindex_base, vindex_tuned, h_base, h_tuned)
    for layer, result in comparison.items():
        base_words = base_tok.batch_decode([[t] for t, _ in result["base_top"][:3]])
        tuned_words = tuned_tok.batch_decode([[t] for t, _ in result["tuned_top"][:3]])
        print(f"  L{layer}: base={base_words}  tuned={tuned_words}")

## Bring in the 1.7B (official tool-calling) for a size comparison

Note: `diff()` requires matching layer counts, so the 1.7B can't be diffed
directly against the 135M models -- it's probe-only here (DESCRIBE + the
tool-call logit-lens check), which is enough to compare *how* tool-calling
shows up at a larger scale vs. how it looks in the small fine-tune above.

In [ ]:
INSTRUCT_1_7B = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

big_tok = AutoTokenizer.from_pretrained(INSTRUCT_1_7B)
big_model = AutoModelForCausalLM.from_pretrained(INSTRUCT_1_7B, torch_dtype=torch.float16).to(device).eval()
vindex_big = extract(big_model, model_name=INSTRUCT_1_7B)
print(vindex_big.num_layers, "layers,", vindex_big.hidden_size, "hidden")

In [ ]:
from marv.probe import logit_lens

start_big = max(1, round(vindex_big.num_layers * 0.66))
probe_layers_big = list(range(start_big, vindex_big.num_layers, 3))
for prompt in DEFAULT_TOOL_PROMPTS:
    print("\nprompt:", prompt)
    h_big = hidden_states_at_layers(big_model, big_tok, prompt, probe_layers_big, device=device)
    for layer, vec in h_big.items():
        idx, logits = logit_lens(vindex_big, vec, k=3)
        print(f"  L{layer}:", big_tok.batch_decode([[t] for t in idx]))

## Swap in the real tool schema

`DEFAULT_TOOL_PROMPTS` above are bare sentences -- they never put the model
in the context it was actually fine-tuned to expect (a system prompt listing
JSON tool schemas, with output constrained to `<tool_call>[...]</tool_call>`).
`marv.toolcall.build_smollm2_tool_prompt` renders that real format (fetched
verbatim from `HuggingFaceTB/SmolLM2-1.7B-Instruct`'s
`instructions_function_calling.md`, not guessed), via the tokenizer's chat
template. The tool-tuned checkpoint's tokenizer doesn't ship its own
`chat_template` even though it shares the base model's vocab -- pass the
base checkpoint's template in explicitly.

In [ ]:
from transformers.utils import get_json_schema
from marv.toolcall import build_smollm2_tool_prompt

def get_weather(location: str) -> str:
    """Gets the current weather for a location.

    Args:
        location: The city to get weather for.
    """
    return "sunny"

tools = [get_json_schema(get_weather)]

TOOL_PROMPT_REAL = build_smollm2_tool_prompt(
    tuned_tok, tools, "What is the weather in Paris?", chat_template=base_tok.chat_template,
)
PLAIN_PROMPT_REAL = build_smollm2_tool_prompt(
    tuned_tok, tools, "Tell me an interesting fact about Paris.", chat_template=base_tok.chat_template,
)
print("token lengths:", len(tuned_tok(TOOL_PROMPT_REAL).input_ids), len(tuned_tok(PLAIN_PROMPT_REAL).input_ids))
print(TOOL_PROMPT_REAL[-160:])

## Cross-reference: did fine-tuning touch the features active at the tool decision point?

`most_changed()` (above) found *which* features fine-tuning moved. Now check
whether those are the *same* features that actually fire when the model is
looking at a real tool-calling prompt -- if fine-tuning's changes and the
model's actual decision-time activity don't overlap, the features `diff()`
flagged may not be causally load-bearing for the behavior.

In [ ]:
from marv.toolcall import firing_features_at_layer, hidden_states_at_layers

changed_by_layer = {}
for d in top:
    changed_by_layer.setdefault(d.layer, set()).add(d.feature_idx)

overlap_layers = sorted(changed_by_layer)
h_tool_real = hidden_states_at_layers(tuned_model, tuned_tok, TOOL_PROMPT_REAL, overlap_layers, device=device)

print(f"{'layer':>5} {'changed features (from diff)':>30} {'firing at tool prompt':>25} {'overlap':>8}")
for layer in overlap_layers:
    changed = changed_by_layer[layer]
    firing = {f for f, _ in firing_features_at_layer(vindex_tuned, h_tool_real[layer], layer, k=20)}
    overlap = changed & firing
    print(f"{layer:>5} {str(sorted(changed)):>30} {str(sorted(firing))[:25]:>25} {str(sorted(overlap)):>8}")

## The big question: do fine-tuning's changed layers match the layers where tool vs. non-tool prompts diverge?

This is the cross-reference between the two notebooks: `diff()`'s per-layer
weight-change ranking (computed here, from the checkpoint pair) against
`marv.layer_heatmap`'s per-layer activation-divergence ranking (tool vs.
plain prompt, on the same tuned model). Run twice -- once with the naive
`DEFAULT_TOOL_PROMPTS`-style bare sentence, once with the real tool-schema
prompt above -- because the answer changes noticeably between them.

In [ ]:
import numpy as np
from marv.diff import per_layer_score
from marv.layer_heatmap import compute as layer_heatmap_compute, difference as layer_heatmap_difference

weight_scores = per_layer_score(deltas, metric="max")
layers = sorted(weight_scores)
w = np.array([weight_scores[l] for l in layers])

def activation_scores_for(tool_prompt, plain_prompt):
    hm_tool = layer_heatmap_compute(vindex_tuned, tuned_model, tuned_tok, tool_prompt, device=device)
    hm_plain = layer_heatmap_compute(vindex_tuned, tuned_model, tuned_tok, plain_prompt, device=device)
    d = layer_heatmap_difference(hm_tool, hm_plain)
    return {layer: float(np.abs(d.matrix[i]).max()) for i, layer in enumerate(d.layers)}

for label, tool_p, plain_p in [
    ("naive bare-sentence prompt", "Call the weather tool", "What's the weather?"),
    ("real tool-schema prompt", TOOL_PROMPT_REAL, PLAIN_PROMPT_REAL),
]:
    activation_scores = activation_scores_for(tool_p, plain_p)
    a = np.array([activation_scores[l] for l in layers])
    corr = np.corrcoef(w, a)[0, 1]
    top_weight = sorted(layers, key=lambda l: -weight_scores[l])[:5]
    top_activation = sorted(layers, key=lambda l: -activation_scores[l])[:5]
    print(f"--- {label} ---")
    print(f"  correlation (weight change vs activation divergence): {corr:.3f}")
    print(f"  top-5 layers by weight change:       {top_weight}")
    print(f"  top-5 layers by activation divergence: {top_activation}")
    print(f"  overlap: {sorted(set(top_weight) & set(top_activation))}")
    print()

## Save a vindex to disk

Skip re-extracting on a future Colab session -- `VindexLite.save()`/`.load()`
round-trip through a plain `.npz`.

In [ ]:
vindex_tuned.save("/content/smollm2_135m_function_calling.npz")

from marv.extract import VindexLite
reloaded = VindexLite.load("/content/smollm2_135m_function_calling.npz")
print(reloaded.num_layers, "layers reloaded,", reloaded.gate[0].shape, "gate shape at layer 0")

## Next steps

- The correlation above is one prompt pair -- rerun `activation_scores_for`
  with several different tool/plain prompt pairs and average, to check the
  0.42-ish correlation (real schema) isn't a one-prompt fluke.
- `notebooks/marv_layer_heatmap_colab.ipynb` adds top-N-per-layer feature
  lists, PCA/t-SNE clustering of several tool vs. plain prompts (with
  literal "tool-cluster feature" extraction), and a combined comparison
  figure -- run that next with `TOOL_PROMPT_REAL`/`PLAIN_PROMPT_REAL` in
  place of its bare-sentence defaults.
- Try the polysemanticity heatmap on a **non-tool-calling** fine-tune pair --
  e.g. swap `BASE_135M`/`TUNED_135M` for `HuggingFaceTB/SmolLM2-135M`
  (pretrained base) vs. `HuggingFaceTB/SmolLM2-135M-Instruct` (SFT+DPO
  instruct-tuned), a completely different, well-documented fine-tune of the
  same architecture -- every cell above runs unchanged.
- A propensity-score-vs-tool-calling-rate scatter (does high mean activation
  on tool-cluster features actually predict the model calling a tool) needs
  a generation loop that labels each prompt with whether the model actually
  emitted a `<tool_call>` -- not built yet, since it's a different kind of
  instrumentation (behavioral, not weight/activation) than everything above.